# Use case 3 — Coverage / gap analysis on Code-typed properties

**Question.** Which `Code`-typed properties have CT bindings, which
don't, and what does each concrete class's effective Code-typed surface
look like (declared + inherited)?

A property is *Code-typed* here if its `rdfs:range` is `usdm:Code`,
`usdm:AliasCode`, or `usdm:ResponseCode` — the three classes for which a
CT binding would make sense. The four `owl:unionOf`-range polymorphic
properties from case 1 are *not* in this set; their union members are
entity classes, not Code subclasses.

**Pinned to v0.1.**


In [1]:
import rdflib
import pandas as pd
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
EXPECTED_BASELINE = 8173

if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/10/20 first to regenerate."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
n_triples = len(g)
print(f"parsed {n_triples} triples from {TTL_PATH}")
if n_triples != EXPECTED_BASELINE:
    print(f"  note: triple count differs from v0.1 baseline ({EXPECTED_BASELINE})"
          " — likely a DDF-RA tag bump; this notebook is pinned to v0.1 binding shape")

def short(uri):
    """Last path segment, fragment-aware."""
    if uri is None:
        return None
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.split("/")[-1]


parsed 8173 triples from ../usdm_v4.ttl


## Step 1 — find all Code-typed properties

SPARQL with a `VALUES` filter on the range. One row per property, with
the v0.1 binding annotations in optional columns.


In [2]:
CODE_TYPED_SPARQL = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?prop ?domain ?range ?boundCodelist ?boundCodelistNote
WHERE {
  ?prop rdfs:domain ?domain ; rdfs:range ?range .
  VALUES ?range { usdm:Code usdm:AliasCode usdm:ResponseCode }
  OPTIONAL { ?prop usdm:boundCodelist ?boundCodelist }
  OPTIONAL { ?prop usdm:boundCodelistNote ?boundCodelistNote }
}
"""

rows = []
for r in g.query(CODE_TYPED_SPARQL):
    prop_short = short(r["prop"])
    rows.append({
        "domain":            short(r["domain"]),
        "prop":              prop_short,
        "attr":              prop_short.split("-", 1)[-1],
        "range":             short(r["range"]),
        "boundCodelist":     short(r["boundCodelist"]),
        "boundCodelistNote": str(r["boundCodelistNote"]) if r["boundCodelistNote"] else None,
    })
code_typed_df = pd.DataFrame(rows).sort_values(["domain", "attr"]).reset_index(drop=True)
print(f"{len(code_typed_df)} Code-typed properties (declared)")
print()
print("by range:")
print(code_typed_df["range"].value_counts().to_string())


67 Code-typed properties (declared)

by range:
range
Code            55
AliasCode       11
ResponseCode     1


## Step 2 — partition by binding status

Three buckets:

- `structured` — has `usdm:boundCodelist` (NCIt IRI for the codelist)
- `free_text`  — has `usdm:boundCodelistNote` only (note text, no extractable C-code)
- `unbound`    — neither annotation present


In [3]:
def bucket_of(row):
    if row["boundCodelist"] is not None and not pd.isna(row["boundCodelist"]):
        return "structured"
    if row["boundCodelistNote"] is not None and not pd.isna(row["boundCodelistNote"]):
        return "free_text"
    return "unbound"

code_typed_df["bucket"] = code_typed_df.apply(bucket_of, axis=1)

breakdown = code_typed_df["bucket"].value_counts().reindex(
    ["structured", "free_text", "unbound"], fill_value=0
)
print("=== bucket counts ===")
print(breakdown.to_string())
print()

for b in ["structured", "free_text", "unbound"]:
    subset = code_typed_df[code_typed_df["bucket"] == b]
    if len(subset) == 0:
        continue
    print(f"--- {b} ({len(subset)}) — first 5 ---")
    print(subset[["domain", "attr", "range", "boundCodelist"]].head(5).to_string(index=False))
    print()


=== bucket counts ===
bucket
structured    45
free_text     12
unbound       10

--- structured (45) — first 5 ---
                      domain                  attr     range boundCodelist
        AdministrableProduct administrableDoseForm AliasCode        C66726
        AdministrableProduct    productDesignation      Code       C207418
        AdministrableProduct              sourcing      Code       C215483
AdministrableProductProperty                  type      Code       C215479
              Administration             frequency AliasCode        C71113

--- free_text (12) — first 5 ---
              domain               attr     range boundCodelist
             Address            country      Code           NaN
AdministrableProduct pharmacologicClass      Code           NaN
     GeographicScope               code AliasCode           NaN
          Indication              codes      Code           NaN
          Ingredient               role      Code           NaN

--- unbound (10)

## Step 3 — inheritance-aware effective coverage per concrete class

For each Concrete class, walk `rdfs:subClassOf*` and gather every
Code-typed property declared on the class itself or any superclass.
The result is each concrete class's *effective* Code-typed surface area.


In [4]:
EFFECTIVE_SPARQL = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?concrete ?declaring ?prop ?range ?boundCodelist ?boundCodelistNote
WHERE {
  ?concrete usdm:modifier "Concrete" .
  ?concrete rdfs:subClassOf* ?declaring .
  FILTER(isIRI(?declaring))
  ?prop rdfs:domain ?declaring ; rdfs:range ?range .
  VALUES ?range { usdm:Code usdm:AliasCode usdm:ResponseCode }
  OPTIONAL { ?prop usdm:boundCodelist ?boundCodelist }
  OPTIONAL { ?prop usdm:boundCodelistNote ?boundCodelistNote }
}
"""

rows = []
for r in g.query(EFFECTIVE_SPARQL):
    prop_short = short(r["prop"])
    bc = short(r["boundCodelist"])
    note = str(r["boundCodelistNote"]) if r["boundCodelistNote"] else None
    if bc:
        bucket = "structured"
    elif note:
        bucket = "free_text"
    else:
        bucket = "unbound"
    rows.append({
        "concrete_class":    short(r["concrete"]),
        "declaring_class":   short(r["declaring"]),
        "prop":              prop_short,
        "attr":              prop_short.split("-", 1)[-1],
        "range":             short(r["range"]),
        "bucket":            bucket,
        "boundCodelist":     bc,
        "boundCodelistNote": note,
    })

effective_df = pd.DataFrame(rows).sort_values(
    ["concrete_class", "declaring_class", "attr"]
).reset_index(drop=True)

CSV_PATH = "code_typed_coverage.csv"
effective_df.to_csv(CSV_PATH, index=False)

print(f"effective coverage: {len(effective_df)} rows × {len(effective_df.columns)} columns")
print(f"distinct concrete classes with Code-typed surface: {effective_df['concrete_class'].nunique()}")
print(f"wrote {CSV_PATH}")
effective_df.head(10)


effective coverage: 72 rows × 8 columns
distinct concrete classes with Code-typed surface: 43
wrote code_typed_coverage.csv


,concrete_class,declaring_class,prop,attr,range,bucket,boundCodelist,boundCodelistNote
0,Address,Address,Address-country,country,Code,free_text,NaN,Y (Point out to ISO 3166-1 Alpha-3 Country code)
1,AdministrableProduct,AdministrableProduct,AdministrableProduct-administrableDoseForm,administrableDoseForm,AliasCode,structured,C66726,Y (SDTM Terminology Codelist C66726)
2,AdministrableProduct,AdministrableProduct,AdministrableProduct-pharmacologicClass,pharmacologicClass,Code,free_text,NaN,"Y (Points to external codelists such as UNII, ..."
3,AdministrableProduct,AdministrableProduct,AdministrableProduct-productDesignation,productDesignation,Code,structured,C207418,Y (C207418)
4,AdministrableProduct,AdministrableProduct,AdministrableProduct-sourcing,sourcing,Code,structured,C215483,Y (C215483)
5,AdministrableProductProperty,AdministrableProductProperty,AdministrableProductProperty-type,type,Code,structured,C215479,Y (C215479)
6,Administration,Administration,Administration-frequency,frequency,AliasCode,structured,C71113,Y (SDTM Terminology Codelist C71113)
7,Administration,Administration,Administration-route,route,AliasCode,structured,C66729,Y (SDTM Terminology Codelist C66729)
8,AliasCode,AliasCode,AliasCode-standardCode,standardCode,Code,unbound,NaN,NaN
9,AliasCode,AliasCode,AliasCode-standardCodeAliases,standardCodeAliases,Code,unbound,NaN,NaN


## Step 4 — gap report

Concrete classes with one or more unbound Code-typed properties on their
effective surface. Sorted by unbound count (descending), then total
surface size.


In [5]:
gap_summary = effective_df.pivot_table(
    index="concrete_class",
    columns="bucket",
    values="prop",
    aggfunc="count",
    fill_value=0,
).reset_index()

for b in ["structured", "free_text", "unbound"]:
    if b not in gap_summary.columns:
        gap_summary[b] = 0

gap_summary["total"] = gap_summary["structured"] + gap_summary["free_text"] + gap_summary["unbound"]
gap_summary = gap_summary[["concrete_class", "structured", "free_text", "unbound", "total"]]

with_gaps = gap_summary[gap_summary["unbound"] > 0].sort_values(
    ["unbound", "total"], ascending=[False, False]
).reset_index(drop=True)

print(f"=== concrete classes with unbound Code-typed properties ===")
print(f"{len(with_gaps)} of {len(gap_summary)} concrete classes have at least one gap")
print()
print(with_gaps.head(15).to_string(index=False))


=== concrete classes with unbound Code-typed properties ===
7 of 43 concrete classes have at least one gap

           concrete_class  structured  free_text  unbound  total
                AliasCode           0          0        2      2
BiomedicalConceptProperty           0          0        2      2
       ExtensionAttribute           0          0        2      2
        BiomedicalConcept           0          0        1      1
BiomedicalConceptCategory           0          0        1      1
        CommentAnnotation           0          0        1      1
             ResponseCode           0          0        1      1


In [6]:
# Drill-down: specific unbound (concrete_class, declaring_class, attr) tuples
unbound_drill = effective_df[effective_df["bucket"] == "unbound"][
    ["concrete_class", "declaring_class", "attr", "range"]
].sort_values(["concrete_class", "declaring_class", "attr"]).reset_index(drop=True)

print(f"=== specific unbound Code-typed property usages ===")
print(f"{len(unbound_drill)} (concrete_class, declaring_class, attr) tuples")
print()
print(unbound_drill.head(20).to_string(index=False))


=== specific unbound Code-typed property usages ===
10 (concrete_class, declaring_class, attr) tuples

           concrete_class           declaring_class                attr        range
                AliasCode                 AliasCode        standardCode         Code
                AliasCode                 AliasCode standardCodeAliases         Code
        BiomedicalConcept         BiomedicalConcept                code    AliasCode
BiomedicalConceptCategory BiomedicalConceptCategory                code    AliasCode
BiomedicalConceptProperty BiomedicalConceptProperty                code    AliasCode
BiomedicalConceptProperty BiomedicalConceptProperty       responseCodes ResponseCode
        CommentAnnotation         CommentAnnotation               codes         Code
       ExtensionAttribute        ExtensionAttribute      valueAliasCode    AliasCode
       ExtensionAttribute        ExtensionAttribute           valueCode         Code
             ResponseCode              Response

## Discussion

What this shows:

- The v0.1 binding layer covers **57 of N** declared Code-typed
  properties (N is the count printed in step 1). Anything in the
  `unbound` bucket is either a real binding gap or a property USDM
  intends to leave open.
- Inheritance multiplies coverage per *concrete class*. A binding
  declared once on a parent (e.g. `PopulationDefinition.plannedSex`)
  contributes effective coverage to every concrete subclass via
  `rdfs:subClassOf*`. Conversely, an unbound declared property
  contributes a gap to every concrete subclass that inherits it.

How to read the gap report:

- Some unbound Code-typed properties are deliberately open-ended in
  USDM design — places where any Code value can appear without
  pre-binding to a single codelist. Don't assume every gap is a v0.1
  oversight. Cross-check against `USDM_CT.xlsx` (the authoritative
  source for *intended* bindings) before classifying.
- The drill-down lists every (concrete_class, declaring_class, attr)
  tuple where a Code-typed property is unbound — useful as a worklist
  for future binding work or as a triage starting point.

What's not shown:

- **Cardinality.** A `0..1` unbound property is a softer gap than a
  required `1` unbound property. This notebook treats them equally;
  case 1's data dictionary carries the cardinality column if you want
  to filter further.
- **Permitted-term coverage.** Whether a *bound* codelist actually has
  permitted terms in CDISC Library is a separate question, answered by
  the deferred Library/CT extension.
